# 平面三连杆：阻抗与接触力控制实验室

这份 Notebook 接在 `01_osc_lab.ipynb` 之后。先观察环境怎样对机器人施力，再完成两个控制律练习：**INT-01 阻抗力**、**INT-02 法向力 PI + 切向位置 PD**。核心代码始终写在 `src/three_link_robot/interaction_controller.py`；这里负责观察、推导核对和运行实验，不复制一份控制器实现。

学习顺序：① 墙与力的符号；② 关节动力学中的外力；③ 自由空间与弹性墙上的阻抗；④ 接触时的法向控力和切向控位；⑤ 对照测试与曲线。缺失函数时会显示待完成提示，不输出伪造轨迹。

## 1. 环境与导入

在仓库根目录安装 `python -m pip install -e '.[dev,notebook]'` 并启动 `python -m jupyterlab`。选择当前仓库的内核。保存 `src/` 中的练习函数后，重新运行对应单元格；需要排除隐藏状态时使用 **Restart Kernel and Run All Cells**。完整公式和单位见 `docs/interaction_learning.md`。

In [ ]:
from pathlib import Path
import sys
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/three_link_robot').is_dir()), None)
if root is None:
    raise RuntimeError('Open this notebook inside the three_link_robot checkout.')
for location in (root, root / 'src'):
    if str(location) not in sys.path:
        sys.path.insert(0, str(location))
%load_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import three_link_robot
from examples.demo_interaction import make_experiment, run_experiment, plot_experiment
from three_link_robot.contact import VerticalWall, contact_state
from three_link_robot.dynamics import forward_dynamics
from three_link_robot.kinematics import forward_kinematics, jacobian
from three_link_robot.interaction_controller import impedance_wrench, hybrid_force_wrench
from three_link_robot.visualization import plot_robot
plt.rcParams.update({'figure.figsize': (7, 4), 'axes.grid': True, 'grid.alpha': .25})
print('Python:', sys.executable)
print('Package:', three_link_robot.__file__)

## 2. 选场景，看清墙在哪里

- `free`：墙在机器人工作空间以外，验证无接触时末端能走到目标。
- `wall`：墙在起点右侧 1 cm，目标却在起点右侧 4 cm，必然出现“目标与墙互相冲突”。
- `force`：墙在起点左侧 1 cm，初始就有约 1.5 N 压力；目标是 2 N，同时保持 `y`。

修改 `TASK` 后，从本单元格开始向下重跑。`initial` 的前六维始终是 `[q, dq]`；`force` 多出的第七维是力误差积分 $z$，单位 N·s。

In [ ]:
TASK = 'free'  # Try 'wall', then 'force'.
DURATION = 5.0
SAMPLES = 151
robot, wall, initial, reference, rhs = make_experiment(TASK)
q0, dq0 = initial[:3], initial[3:6]
xy0, dxy0, f_env0 = contact_state(q0, dq0, robot, wall)
print('Scenario:', TASK)
print('Initial q [rad]:', q0)
print('Initial endpoint [m]:', xy0)
print('Wall x [m]:', wall.x)
print('Penetration [m]:', wall.penetration(xy0))
print('Wall-on-robot force [N]:', f_env0)
print('Robot-on-wall normal force [N]:', wall.force_on_wall(xy0, dxy0))
print('Reference:', reference(0.0))
print('Initial state shape:', initial.shape)

### 初始构型、墙和目标

下图只显示几何关系；尚未运行控制器。接触场景中的黑色竖线是墙；`free` 的墙在工作空间外，所以图中省略。机器人的末端可压入这面**有限刚度**的墙，压入量会转化为接触力；它并非绝对不可穿过的刚体约束。

In [ ]:
ax, _ = plot_robot(forward_kinematics(q0, robot), reach=sum(robot.lengths))
if TASK != 'free':
    ax.axvline(wall.x, color='black', linestyle='--', label='Compliant wall')
if TASK == 'force':
    ax.axhline(reference(0.0).y, color='tab:orange', linestyle=':', label='Desired y')
else:
    desired = reference(0.0).x
    ax.plot(desired[0], desired[1], 'x', color='tab:orange', markersize=10, label='Desired xy')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title('Initial geometry — no closed-loop simulation yet')
ax.legend()
plt.show()

## 3. 理解单边弹性墙

末端从左侧沿 `+x` 压墙。墙作用于机器人的力 $f_{env,x}$ 为负；机器人作用于墙的法向力大小 $f_n=-f_{env,x}$ 为正。压入量 $\delta=\max(0,x-x_{wall})$。只有 $\delta>0$ 才能接触；墙不会在末端尚未碰到它时施力，也不会拉住正在脱离的末端。

下面把 `y` 和速度固定，只改变 `x`，直接画出**墙模型**的力曲线。它不依赖 INT-01/02，适合先检查力的方向和单位。

In [ ]:
probe_x = np.linspace(wall.x - .02, wall.x + .03, 101)
probe_fn = np.array([wall.force_on_wall([x, xy0[1]], [0., 0.]) for x in probe_x])
fig, ax = plt.subplots()
ax.plot(1000 * (probe_x-wall.x), probe_fn)
ax.axvline(0., color='black', linestyle='--')
ax.set(xlabel='x - wall [mm]', ylabel='Force on wall [N]',
       title='Unilateral spring at zero endpoint velocity')
plt.show()
print('Spring stiffness [N/m]:', wall.stiffness)
print('Expected at 10 mm penetration [N]:', wall.stiffness * .01)

### 环境力加在哪个动力学方程中？

原项目的 `forward_dynamics(q,dq,tau,robot)` 解 $M\ddot q+h=\tau$。接触时要把外力映射成 $J^Tf_{env}$ 后加到右侧：

$$
M\ddot q+h=\tau_{ctrl}+J^Tf_{env}.
$$

这一步在 `contact.py` 的状态方程中完成，**不是**在 `impedance_wrench` 中加一次力。下面只比较同一关节状态下，有无墙力导致的加速度差。

In [ ]:
J0 = jacobian(q0, robot)
tau_probe = np.zeros(3)
ddq_free = forward_dynamics(q0, dq0, tau_probe, robot)
ddq_contact = forward_dynamics(q0, dq0, tau_probe + J0.T @ f_env0, robot)
print('J.T @ f_env [N·m]:', J0.T @ f_env0)
print('No-contact ddq [rad/s²]:', ddq_free)
print('With-wall ddq [rad/s²]:', ddq_contact)
print('Difference [rad/s²]:', ddq_contact-ddq_free)

## 4. INT-01：阻抗如何让末端“软下来”？

填写 `impedance_wrench`，让位置和速度误差产生二维任务力：

$$
F_{imp}=K(x_d-x)+D(\dot x_d-\dot x).
$$

本实验的任务空间加速度前馈与完整 $h=c+g$ 补偿已经在外围接好。令 $e=x-x_d$，闭环应满足 $\Lambda(q)\ddot e+D\dot e+Ke=f_{env}$。这说明墙力会造成**可预测的位置偏移**，而不是控制器“失效”。此处表观惯量是机器人实际的 $\Lambda(q)$，尚未做常数期望惯量的整形。

先运行下方练习探针。若看到 `INT-01 待完成`，去 `src/three_link_robot/interaction_controller.py` 填公式并保存，然后重跑。

In [ ]:
if TASK == 'force':
    print('这个场景先看 INT-02；将 TASK 改为 free 或 wall 后检查 INT-01。')
else:
    ref0 = reference(0.0)
    try:
        wrench = impedance_wrench(xy0, dxy0, ref0,
                                  np.array([60., 60.]), np.array([20., 20.]))
    except NotImplementedError as error:
        print('INT-01 待完成：', error)
    else:
        print('Your impedance task wrench [N]:', wrench)

### 碰墙时，先手算最终会有多大力

忽略稳态速度后，机器人虚拟弹簧与墙弹簧平衡：

$$
K_x(x_d-x)=k_w(x-x_{wall}),\qquad
f_{n,\infty}=\frac{K_x k_w}{K_x+k_w}(x_d-x_{wall}).
$$

`wall` 场景中，目标比墙深入 3 cm，$K_x=60$ N/m、$k_w=150$ N/m，预测约 1.286 N。**先写下你的预测**：将 $K_x$ 加倍，最终压力会增大还是减小？再改实验增益验证。注意修改增益需要修改 `examples/demo_interaction.py` 的场景配置。

In [ ]:
if TASK == 'wall':
    Kx = 60.0
    desired_x = reference(0.0).x[0]
    extra_depth = desired_x - wall.x
    expected_static_force = Kx * wall.stiffness / (Kx + wall.stiffness) * extra_depth
    print('Desired position beyond wall [m]:', extra_depth)
    print('Predicted steady contact force [N]:', expected_static_force)
else:
    print('切换 TASK=wall 可计算稳态力的理论预测。')

## 5. INT-02：法向力 PI 与切向位置 PD

`force` 场景从**已有接触**的状态开始。测得的 $f_m=-f_{env,x}\ge0$，期望施力 $f_d=2$ N，积分状态 $z=\int(f_d-f_m)dt$。你要把法向和切向目标变成两个任务力分量：

$$
u_n=f_d+K_{pf}(f_d-f_m)+K_{if}z,\qquad
u_y=K_{py}(y_d-y)-K_{dy}\dot y.
$$

外围将 $[u_n,u_y]^T$ 用 $J^T$ 映射到关节力矩，并补偿 $h$。接触丢失时冻结 $z$，避免持续积分不可实现的力误差。这个弹性墙模型没有严格刚性接触约束；法向和切向的运动仍可能被任务惯量耦合。

In [ ]:
if TASK != 'force':
    print('切换 TASK=force 后检查法向控力。')
else:
    ref0 = reference(0.0)
    f_measured = wall.force_on_wall(xy0, dxy0)
    print('Desired / initial measured normal force [N]:', ref0.force_n, f_measured)
    print('Initial force error [N]:', ref0.force_n - f_measured)
    print('Initial y error [m]:', ref0.y-xy0[1])
    try:
        wrench = hybrid_force_wrench(ref0.force_n, f_measured, ref0.y, xy0[1],
                                      dxy0[1], initial[6], kp_force=.5,
                                      ki_force=1.0, kp_y=80., kd_y=16.)
    except NotImplementedError as error:
        print('INT-02 待完成：', error)
    else:
        print('Your task wrench [normal, tangent] N:', wrench)

## 6. 运行并读图

`free` / `wall` 需要 INT-01；`force` 需要 INT-02。图中依次是末端路径与墙、法向力、压入量、切向位置。`wall` 应靠近手算的稳态力；`force` 应靠近 2 N 并维持 `y`。运行前先求一次状态导数，确保未填练习不会进入积分。其他代码错误会直接显示，方便排查。

In [ ]:
data = None  # Do not reuse old results after a failed rerun.
try:
    rhs(0.0, initial)
except NotImplementedError as error:
    print('尚未运行闭环：', error)
else:
    data = run_experiment(TASK, duration=DURATION, samples=SAMPLES)
    fig = plot_experiment(data)
    plt.show()

### 用数字核对，而不只看曲线

`free` 的力应始终为零；`wall` 的稳态力应与弹簧平衡预测接近；`force` 的稳态力应接近 2 N，且 `y` 偏差很小。若压入量一直为负，说明根本没有建立接触，不能讨论“力跟踪”。

In [ ]:
if data is None:
    print('没有闭环数据；先完成对应 INT 练习。')
else:
    print(f"Final normal force [N]: {data['force'][-1]:.6g}")
    print(f"Peak normal force [N]: {data['force'].max():.6g}")
    print(f"Final penetration [mm]: {1000*max(0., data['xy'][-1,0]-wall.x):.6g}")
    if TASK == 'force':
        print(f"Final force error [N]: {reference(DURATION).force_n-data['force'][-1]:.6g}")
        print(f"Final y error [m]: {reference(DURATION).y-data['xy'][-1,1]:.6g}")
    elif TASK == 'wall':
        print(f"Predicted final force [N]: {expected_static_force:.6g}")
        print(f"Difference [N]: {data['force'][-1]-expected_static_force:.6g}")

## 7. 自测与下一步

在仓库根目录运行：

```bash
python -m pytest tests/test_interaction.py -q -rs
python -m examples.demo_interaction --task free
python -m examples.demo_interaction --task wall
python -m examples.demo_interaction --task force
```

测试显示 `skipped` 只表示练习未完成。判断控制器前，先确认墙模型、外力符号和实际接触状态。完成后可依次试：改变墙刚度、取消力积分、加入被控对象与控制器参数失配。后两项需要额外设计实验；当前“理想模型下误差很小”不等于真实机器人鲁棒。